In [22]:
!pip install -q "langchain==0.2.16" "langchain-core==0.2.43" "langchain-groq"

### **Basic LLM Call**

In [23]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq

# Securely fetching the key from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# Initialize the Llama-3.3-70B model
llm=ChatGroq(model="llama-3.3-70b-versatile",temperature=0.5)

try:
    response=llm.invoke("What are the 3 core principles of LangChain?")
    print(response.content)
except Exception as e:
    print(f"Connection Error: {e}")

LangChain is a framework for building applications that utilize large language models. The three core principles of LangChain are:

1. **Modularity**: Breaking down complex applications into smaller, modular components that can be easily integrated and reused. This allows developers to build and test individual components independently, making it easier to maintain and update their applications.

2. **Composability**: Enabling the combination of multiple components, including language models, to create more powerful and flexible applications. This allows developers to leverage the strengths of different models and components to achieve specific goals.

3. **Chainability**: Allowing components to be chained together in a flexible and dynamic way, enabling the creation of complex workflows and applications. This enables developers to build applications that can adapt to changing requirements and user needs.

By following these principles, LangChain aims to simplify the process of buildin

### **PromptTemplate Usage**

In [24]:
from langchain_core.prompts import PromptTemplate

prompt=PromptTemplate.from_template("Explain {topic} in simple terms")
formatted_prompt=prompt.format(topic="GenAI")

print(formatted_prompt)
print("\n--- LLM Output ---\n")
print(llm.invoke(formatted_prompt).content)

Explain GenAI in simple terms

--- LLM Output ---

GenAI, short for General Artificial Intelligence, refers to a type of artificial intelligence that can perform a wide range of tasks, similar to how humans do. It's like a super-smart computer that can:

1. **Learn**: GenAI can learn from data, experiences, and interactions, just like humans.
2. **Reason**: It can understand and make decisions based on logic, patterns, and context.
3. **Apply**: GenAI can apply its knowledge and skills to various tasks, such as problem-solving, creating, and communicating.
4. **Improve**: It can continuously learn and improve its performance over time, adapting to new situations and challenges.

Imagine a computer that can:

* Understand and respond to natural language (like a human conversation)
* Recognize and create images, music, and videos
* Play games, solve puzzles, and make decisions
* Learn from data and experiences to improve its performance

GenAI is often compared to human intelligence beca

### **Simple Chain (LCEL)**

In [25]:
from langchain_core.output_parsers import StrOutputParser

# Create the Chain
chain=prompt_template | llm | StrOutputParser()

# Run the Chain
result = chain.invoke({"concept": "Tokenization","audience": "Backend Developer"})
print("\n--- Chain Output ---")
print(result)


--- Chain Output ---
**Tokenization: Breaking Down Text into Manageable Pieces**

Imagine you're a chef, and you have a long sentence written on a piece of paper: "I love to eat delicious pizzas on Fridays." To work with this sentence, you need to break it down into individual words or "ingredients" that you can analyze and process.

Tokenization is like chopping the sentence into individual words or tokens: "I", "love", "to", "eat", "delicious", "pizzas", "on", "Fridays". This process makes it easier to analyze, process, and understand the meaning of the text.

In NLP, tokenization is the first step in text processing, allowing you to work with individual words, phrases, or characters, rather than a long, unwieldy sentence. This enables you to perform tasks like sentiment analysis, named entity recognition, and language modeling more efficiently.

**Example Use Case:**

* Input: "I love to eat delicious pizzas on Fridays."
* Tokenization Output: ["I", "love", "to", "eat", "delicious"

### **Agent with Tool**

In [26]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

# Model
llm=ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Tool
@tool
def calculate_checkout_total(quantity: int, price_per_unit: float) -> str:
    """Calculates the final checkout price including a ₹97 shipping fee."""
    shipping=97
    subtotal=quantity * price_per_unit
    grand_total=subtotal + shipping
    return f"Subtotal: ₹{subtotal}. Grand Total: ₹{grand_total} (includes ₹{shipping} shipping)."

tools=[calculate_checkout_total]
tool_map={t.name: t for t in tools}

# Bind tools to model
llm_with_tools=llm.bind_tools(tools)

# First call
messages=[HumanMessage("Calculate the total for 3 items at ₹300 each.")]
response=llm_with_tools.invoke(messages)
messages.append(response)

# Execute tool calls
for tool_call in response.tool_calls:
    result=tool_map[tool_call["name"]].invoke(tool_call["args"])
    messages.append(ToolMessage(result, tool_call_id=tool_call["id"]))

# Final answer
final=llm_with_tools.invoke(messages)
print("Final Output:", final.content)

Final Output: The total for 3 items at ₹300 each is ₹997.


### **Memory example**

In [27]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Prompt with a messages placeholder for history
memory_prompt=ChatPromptTemplate.from_messages([("system", "You are a helpful coding assistant."),MessagesPlaceholder(variable_name="chat_history"),("human", "{input}"),])

# Build the chain (LCEL style)
chain=memory_prompt | llm

# Simple in-memory store (session_id → history object)
store={}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id]=InMemoryChatMessageHistory()
    return store[session_id]

# Wrap chain with history management
conversation=RunnableWithMessageHistory(chain,get_session_history,input_messages_key="input",history_messages_key="chat_history",)

# Config ties messages to a specific session
config={"configurable": {"session_id": "my_session"}}

# Turn 1
conversation.invoke(
    {"input": "Hi! I'm learning LangChain to build an NLP blog."},
    config=config
)

# Turn 2 — model remembers the previous message
response = conversation.invoke(
    {"input": "What should be my first technical chapter?"},
    config=config
)

print("\n--- Memory-Aware Response ---")
print(response.content)


--- Memory-Aware Response ---
For your first technical chapter, I'd recommend starting with the basics of LangChain and setting up a foundation for your NLP blog. Here are a few options:

1. **Introduction to LangChain and its Components**: In this chapter, you could introduce the core concepts of LangChain, such as the Agent, Index, and Chain. You could also cover the different types of agents, like the LLM (Large Language Model) agent, and how they interact with each other.
2. **Setting up a LangChain Environment**: This chapter could focus on setting up a development environment for LangChain, including installing the necessary dependencies, configuring the environment, and running a simple "Hello World" example.
3. **Building a Simple NLP Pipeline**: In this chapter, you could guide readers through building a simple NLP pipeline using LangChain, such as text classification or sentiment analysis. This would help readers understand how to integrate LangChain with other NLP libraries